# Materials 04 — Building a grain boundary

Real materials are made of many grains separated by **grain boundaries**
(GBs), whose structure and energy control mechanical and transport
properties. Following the original Tutorial 4, we construct the classic
**Σ5(310) symmetric tilt boundary** in our fcc LJ metal and compute its
energy.

The construction: two half-crystals, rotated by equal and opposite angles
about a common [100] tilt axis, meeting on a (310) plane. The box must be
**commensurate** with the boundary's periodicity — for Σ5(310) the repeat
along the boundary is a₀√10/2, so box lengths are chosen as whole multiples
of it (an incommensurate box forces a strained, artificially high-energy
boundary — the check in the original tutorial's yellow box).

In [ ]:
%pip install lammps-js matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lammps import lammps, LMP_STYLE_ATOM, LMP_TYPE_VECTOR

A0, ECOH = 1.5496, -8.0998            # from tutorial 01
P = A0 * np.sqrt(10) / 2              # CSL period along x and y
LX, LY, LZ = 4 * P, 4 * P, 2 * A0     # commensurate box

BICRYSTAL = f"""
units lj
atom_style atomic
boundary p p p
lattice fcc {4 / A0**3:.8f}
region box block 0 {LX:.6f} -{LY:.6f} {LY:.6f} 0 {LZ:.6f} units box
create_box 2 box
lattice fcc {4 / A0**3:.8f} orient x 0 3 1 orient y 0 -1 3 orient z 1 0 0
region upper block INF INF 0.0 INF INF INF units box
create_atoms 1 region upper
lattice fcc {4 / A0**3:.8f} orient x 0 3 -1 orient y 0 1 3 orient z 1 0 0
region lower block INF INF INF 0.0 INF INF units box
create_atoms 2 region lower
mass * 1.0
pair_style lj/cut 2.5
pair_coeff * * 1.0 1.0 2.5
group upper type 1
group lower type 2
"""

async def gb_energy(overlap_cutoff):
    """Build the bicrystal, delete near-coincident atoms, relax, return (gamma, lmp)."""
    lmp = await lammps(output=None)
    lmp.commands_string(BICRYSTAL)
    lmp.command(f"delete_atoms overlap {overlap_cutoff} lower upper")
    lmp.commands_string("""
min_style cg
minimize 1.0e-10 1.0e-10 5000 50000
fix relax all box/relax y 0.0 vmax 0.001
minimize 1.0e-10 1.0e-10 5000 50000
""")
    n = lmp.get_natoms()
    etot = lmp.get_thermo("pe") * n            # units lj: pe is per atom
    area = lmp.get_thermo("lx") * lmp.get_thermo("lz")
    gamma = (etot - n * ECOH) / (2 * area)     # 2 boundaries in a periodic box
    return gamma, n, lmp

## The overlap cutoff is a *structural search parameter*

Where the grains meet, atoms from opposite sides can nearly coincide and
must be removed (`delete_atoms overlap`). Different cutoffs produce
**different boundary structures with different energies** — sampling them and
keeping the lowest is the simplest version of a real GB-structure search:

In [ ]:
cutoffs = [0.15, 0.3, 0.45, 0.55, 0.7, 0.85]
gammas, best = [], None
for cut in cutoffs:
    g, n, inst = await gb_energy(cut)
    gammas.append(g)
    if cut == 0.7:
        best = inst          # keep the relaxed plateau structure for later
    else:
        inst.close()
    print(f"cutoff {cut:4} -> {n:4d} atoms, gamma_GB = {g:.4f} eps/sigma^2")
lmp = best

plt.figure(figsize=(5.5, 3.2))
plt.plot(cutoffs, gammas, "o-")
plt.xlabel("overlap deletion cutoff (σ)"); plt.ylabel("γ_GB (ε/σ²)")
plt.tight_layout(); plt.show()

Small cutoffs keep the near-coincident pairs → a high-energy, jammed
boundary. Past ~0.5 σ the doubled atoms are removed and the energy drops to
a plateau — that plateau is our Σ5(310) boundary energy. (A definitive value
would also scan rigid translations of one grain — the same caveat as the
original tutorial.)

## See the boundary

No 3-D viewer needed: color each atom by its **centrosymmetry parameter**
and project on the x–y plane. The periodic *structural units* of the
boundary light up, just like Figure 7 of the paper — note the two boundaries
(the box is periodic in y):

In [ ]:
lmp.command("compute csym all centro/atom fcc")
lmp.command("run 0 post no")
csym = lmp.extract_compute("csym", LMP_STYLE_ATOM, LMP_TYPE_VECTOR)
x = lmp.extract_atom("x")

plt.figure(figsize=(6, 5))
sc = plt.scatter(x[:, 0], x[:, 1], c=csym, s=28, cmap="coolwarm")
plt.colorbar(sc, label="centrosymmetry")
plt.xlabel("x (σ)"); plt.ylabel("y (σ)")
plt.title("Σ5(310) tilt boundary, colored by centrosymmetry")
plt.gca().set_aspect("equal")
plt.tight_layout(); plt.show()

lmp.close()

**Exercises**
- Make the box twice as tall (`LY = 8 * P`). γ_GB should not change — if it
  does, the boundaries are interacting through the periodic images.
- Try an *incommensurate* box (e.g. `LX = 4.5 * P`) and watch γ_GB jump.

Next: [05 — Grain-boundary fracture](05-fracture.ipynb) — boundaries are
where materials break.